In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma youtube-transcript-api rank_bm25 gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [10]:
from getpass import getpass
import os

print("--- Hugging Face Setup ---")
if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Enter your Hugging Face Access Token: ")

print("\n--- LangSmith Evaluation Setup (Optional) ---")
use_langsmith = input("Do you want to enable LangSmith tracing? (y/n): ")
if use_langsmith.lower() == 'y':
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LangSmith API Key: ")
    os.environ["LANGCHAIN_PROJECT"] = "Advanced_YouTube_RAG"
    print("LangSmith Tracing Enabled!")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"

--- Hugging Face Setup ---

--- LangSmith Evaluation Setup (Optional) ---
Do you want to enable LangSmith tracing? (y/n): n


In [4]:
"""
YouTube RAG Assistant

A decoupled retrieval-augmented generation (RAG) system utilizing
a custom hybrid retrieval layer and the native Hugging Face Inference API.

Dependencies:
    pip install langchain-community langchain-chroma langchain-huggingface gradio huggingface_hub rank_bm25 youtube-transcript-api
"""

import os
import gradio as gr
from langchain_community.document_loaders.youtube import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from huggingface_hub import InferenceClient

# Disable LangSmith tracing warning flags
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Global runtime state tracker
app_state = {"retriever": None}


class CustomHybridRetriever:
    """
    A lightweight, stable replacement for EnsembleRetriever.
    Combines keyword searches (BM25) and semantic vector searches (Chroma).
    """
    def __init__(self, bm25_retriever, vector_retriever):
        self.bm25 = bm25_retriever
        self.vector = vector_retriever

    def invoke(self, query: str):
        # Fetch matching segments from both retrieval styles
        bm25_docs = self.bm25.invoke(query)
        vector_docs = self.vector.invoke(query)

        # Merge results and deduplicate by document content
        seen_contents = set()
        combined_docs = []

        for doc in (vector_docs + bm25_docs):
            if doc.page_content not in seen_contents:
                seen_contents.add(doc.page_content)
                combined_docs.append(doc)

        return combined_docs


def ask_llm(context, question):
    """
    Queries the Hugging Face Inference API with a fallback structure.
    Handles user intent nuances like channel cast identities and
    linguistic parsing for 'characters'.
    """
    client = InferenceClient()

    prompt = f"""You are an expert AI assistant analyzing a YouTube video transcript.

Instructions:
1. Use the provided context to accurately answer the question.
2. Use basic common sense reasoning to connect well-known names to channel identities (e.g., if the context mentions Jimmy, Chandler, and Tareq, it refers to a MrBeast video).
3. If the user asks about "characters", assume they mean the people or cast members present in the video, not the text string character length.
4. If the answer cannot be inferred from the context at all, state that you do not know.

Context:
{context}

Question:
{question}

Answer:"""

    try:
        # Default endpoint query to Mistral
        response = client.chat.completions.create(
            model="mistralai/Mistral-7B-Instruct-v0.3",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception:
        try:
            # Automatic traffic failover to Llama
            response = client.chat.completions.create(
                model="meta-llama/Llama-3.1-8B-Instruct",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512,
                temperature=0.1
            )
            return response.choices[0].message.content
        except Exception as fallback_error:
            return f"Error: Both inference endpoints failed. Details: {fallback_error}"

def build_rag_pipeline(video_url):
    """
    Downloads transcript, chunks text, and indexes into a
    hybrid dense-sparse (Chroma + BM25) vector space.
    """
    loader = YoutubeLoader.from_youtube_url(video_url, add_video_info=False)
    try:
        docs = loader.load()
    except Exception as e:
        return None, f"Error loading transcript: {e}"

    if not docs or not docs[0].page_content.strip():
        return None, "Error: Transcript is empty or could not be verified."

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

    vector_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 10})
    bm25_retriever = BM25Retriever.from_documents(splits)
    bm25_retriever.k = 3

    # Instantiate our custom tool wrapper instead of LangChain's EnsembleRetriever
    hybrid_retriever = CustomHybridRetriever(bm25_retriever, vector_retriever)

    return hybrid_retriever, "Success: Video processed and custom hybrid retriever ready."

def process_video_ui(url, token):
    """
    Validates user configurations and binds the retriever instantiation
    to the global environment state.
    """
    if not url.strip():
        return "Error: Please provide a valid YouTube URL."
    if not token.strip():
        return "Error: Please provide your Hugging Face API token."

    os.environ["HF_TOKEN"] = token.strip()
    retriever, status_msg = build_rag_pipeline(url)

    if retriever:
        app_state["retriever"] = retriever
    return status_msg

def chat_interface_logic(message, history):
    """Main routing function execution step for the Gradio ChatInterface module."""
    if not app_state["retriever"]:
        return "Error: Please process a YouTube video with a valid token first."

    try:
        retriever = app_state["retriever"]
        retrieved_docs = retriever.invoke(message)
        context = "\n\n".join(doc.page_content for doc in retrieved_docs)

        return ask_llm(context, message).strip()
    except Exception as e:
        return f"Processing Error: {e}"

# Gradio UX configuration and rendering layout block
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# YouTube RAG Assistant")
    gr.Markdown("A decoupled local retrieval system utilizing local storage and the native Hugging Face Inference API.")

    with gr.Row():
        url_input = gr.Textbox(label="YouTube URL", placeholder="https://www.youtube.com/watch?v=...", scale=3)
        token_input = gr.Textbox(label="Hugging Face Token", placeholder="hf_...", type="password", scale=2)
        process_btn = gr.Button("Process Video", variant="primary", scale=1)

    status_output = gr.Textbox(label="System Status", interactive=False)
    process_btn.click(fn=process_video_ui, inputs=[url_input, token_input], outputs=status_output)

    gr.ChatInterface(fn=chat_interface_logic)

if __name__ == "__main__":
    demo.launch(debug=True)

/tmp/ipykernel_16957/299012745.py:160: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e3b7fae0882a74cd6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e3b7fae0882a74cd6a.gradio.live


In [17]:
!pip install -q langchain-classic

In [19]:
pip install langchain rank_bm25

In [28]:
!pip install --upgrade --force-reinstall langchain langchain-core langchain-community

  Using cached langchain_core-1.4.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.1 MB/s eta 0:00:00
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.2 MB/s eta 0:00:00
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 8.6 MB/s eta 0:00:00
Using cached langchain_core-1.4.1-py3-none-any.whl (549 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.9 MB/s eta 0:00:00
Using cached langchain_classi

In [2]:
pip install youtube-transcript-api